# 데이터 누출 실습

**Data Leakage · 정보 누출**

평가 데이터의 정보가 학습 과정에 흘러들어 성능이 실제보다 좋게 보이는 문제.

소재 분야에서 이해하기: 전체 데이터로 스케일링을 먼저 하면 시험 데이터 정보가 새어든다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 교차검증 문서](https://scikit-learn.org/stable/modules/cross_validation.html)

## 1. 전처리를 전체 데이터에 먼저 적용하면

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

from sklearn.model_selection import cross_val_score, KFold
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

# 실제로는 전부 잡음이라 예측이 불가능한 데이터
n, p = 80, 3000
X_noise = rng.normal(0, 1, (n, p))
y_noise = rng.normal(0, 1, n)
print('입력과 목표는 서로 독립입니다. 정직한 R2는 0 근처여야 합니다.')

In [ ]:
# 잘못된 방법: 전체 데이터로 변수를 먼저 고른 뒤 교차검증
selected = SelectKBest(f_regression, k=20).fit_transform(X_noise, y_noise)
leaked = cross_val_score(LinearRegression(), selected, y_noise,
                         cv=KFold(5, shuffle=True, random_state=0), scoring='r2').mean()

# 올바른 방법: 변수 선택을 파이프라인에 넣어 폴드 안에서만 수행
pipeline = make_pipeline(SelectKBest(f_regression, k=20), LinearRegression())
honest = cross_val_score(pipeline, X_noise, y_noise,
                         cv=KFold(5, shuffle=True, random_state=0), scoring='r2').mean()
print('전체 데이터로 변수 선택 후 교차검증 R2 %.3f  <- 누출' % leaked)
print('파이프라인 안에서 변수 선택   R2 %.3f  <- 정직' % honest)

## 2. 스케일링도 같은 문제를 만듭니다

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor

X_real, y_real, _ = None, None, None
X_real = rng.uniform(0, 1, (200, 5))
y_real = X_real[:, 0] * 3 + rng.normal(0, 0.2, 200)

scaled_first = StandardScaler().fit_transform(X_real)      # 전체로 fit (누출)
leaked = cross_val_score(LinearRegression(), scaled_first, y_real, cv=5, scoring='r2').mean()
honest = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()), X_real, y_real,
                         cv=5, scoring='r2').mean()
print('전체로 스케일링 후 교차검증 R2 %.4f' % leaked)
print('파이프라인 스케일링       R2 %.4f' % honest)
print('\n차이가 작아 보여도, 표본이 적거나 이상치가 있으면 커집니다.')
print('원칙: 학습 데이터에서만 fit 하고, 검증·시험에는 transform 만 적용합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#data-leakage)을 여세요.